# 02-05 레이어를 쌓아 모델 만들기

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 데이터: 나선 세 개

In [ ]:
def make_spirals(n=300, classes=3, noise=0.15):
    torch.manual_seed(0)
    xs, ys = [], []
    for c in range(classes):
        r = torch.linspace(0.1, 1, n)
        t = torch.linspace(0, 3.5, n) + c * 2 * math.pi / classes + noise * torch.randn(n)
        xs.append(torch.stack([r * torch.sin(t), r * torch.cos(t)], 1))
        ys.append(torch.full((n,), c))
    return torch.cat(xs), torch.cat(ys)

X, Y = make_spirals()  # (900, 2), (900,)
plt.figure(figsize=(4, 4))
plt.scatter(X[:, 0], X[:, 1], c=Y, cmap="brg", s=8)
plt.gca().set_aspect("equal")
plt.show()
X, Y = X.to(device), Y.to(device)

## 2. nn.Sequential로 쌓기

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 64),  # (batch, 2) -> (batch, 64)
    nn.ReLU(),
    nn.Linear(64, 64),  # (batch, 64) -> (batch, 64)
    nn.ReLU(),
    nn.Linear(64, 3),  # (batch, 64) -> (batch, 3)
)
print(model)

In [ ]:
x = torch.rand(5, 2)  # (5, 2)
for layer in model:
    x = layer(x)
    print(f"{layer.__class__.__name__:8s} -> {tuple(x.shape)}")

## 3. 블록 만들기: 정규화, 드롭아웃, 잔차 연결

In [ ]:
class Block(nn.Module):
    """LayerNorm -> Linear -> ReLU -> Dropout, 그리고 입력을 더하는 블록"""

    def __init__(self, d, dropout=0.1, residual=True):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.linear = nn.Linear(d, d)
        self.dropout = nn.Dropout(dropout)
        self.residual = residual

    def forward(self, x):
        # x: (batch, d)
        h = self.dropout(torch.relu(self.linear(self.norm(x))))  # (batch, d)
        return x + h if self.residual else h  # (batch, d)


class MLP(nn.Module):
    def __init__(self, d=64, n_blocks=4, n_classes=3, residual=True):
        super().__init__()
        self.inp = nn.Linear(2, d)
        self.blocks = nn.ModuleList([Block(d, residual=residual) for _ in range(n_blocks)])
        self.out = nn.Linear(d, n_classes)

    def forward(self, x):
        x = self.inp(x)  # (batch, 2) -> (batch, d)
        for block in self.blocks:
            x = block(x)  # (batch, d) -> (batch, d)
        return self.out(x)  # (batch, d) -> (batch, n_classes)

model = MLP()
print(model)
print("파라미터 수:", sum(p.numel() for p in model.parameters()))

## 4. 학습하기

In [ ]:
def train(model, steps=1500, lr=1e-3):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    model.train()
    for step in range(steps):
        loss = F.cross_entropy(model(X), Y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    model.eval()
    with torch.no_grad():
        acc = (model(X).argmax(-1) == Y).float().mean().item()
    return losses, acc

torch.manual_seed(0)
model = MLP(n_blocks=4)
losses, acc = train(model)
print(f"블록 4개: 마지막 손실 {losses[-1]:.4f}, 정확도 {acc:.3f}")

In [ ]:
g = torch.linspace(-1.2, 1.2, 300)
gx, gy = torch.meshgrid(g, g, indexing="xy")
pts = torch.stack([gx.flatten(), gy.flatten()], 1).to(device)  # (90000, 2)
with torch.no_grad():
    pred = model(pts).argmax(-1).reshape(300, 300).cpu()  # (300, 300)
plt.figure(figsize=(4.5, 4.5))
plt.contourf(gx, gy, pred, levels=[-0.5, 0.5, 1.5, 2.5], cmap="brg", alpha=0.3)
plt.scatter(X[:, 0].cpu(), X[:, 1].cpu(), c=Y.cpu(), cmap="brg", s=6)
plt.gca().set_aspect("equal")
plt.title(f"4 blocks, acc={acc:.3f}")
plt.show()

## 5. 잔차 연결이 왜 필요할까

In [ ]:
curves = {}
for residual in [False, True]:
    torch.manual_seed(0)
    m = MLP(n_blocks=20, residual=residual)
    losses, acc = train(m, steps=800)
    curves[residual] = losses
    print(f"블록 20개, 잔차 연결 {residual}: 100단계 손실 {losses[100]:.4f}, 마지막 손실 {losses[-1]:.4f}, 정확도 {acc:.3f}")

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(curves[False], color="tab:gray", label="20 blocks, no residual")
plt.plot(curves[True], color="tab:orange", label="20 blocks, residual")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()